# 🥈 02 – Silver : nettoyage & transformation (PySpark)

1. Remplacement des valeurs `"NA"` par `null` et suppression des doublons
2. Création d'une clé technique `Participation_ID`
3. Harmonisation des noms d'équipes à partir du code **NOC** (mapping NOC → pays)
4. Correction de `Year` / `Season` à partir de la colonne `Games`, nettoyage de `Age`, calcul de `Birth_Year`
5. Typage de `Height` / `Weight`
6. Ajout des indicateurs `hasParticipated` et `Medal_flag`
7. Écriture en **Delta** dans `silver/athletes_transformed_data/`

In [ ]:
%run ./00_auth

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lit, col, when, split

In [ ]:
data = spark.read.format("parquet").load("abfss://bronze@mohammedstorageacc2.dfs.core.windows.net/")

In [ ]:
data = data.replace("NA", None)

In [ ]:
data=data.dropDuplicates()

In [ ]:
w = Window.orderBy(lit(1))
data = data.withColumn("Participation_ID", row_number().over(w) - 1)
display(data)

In [ ]:
mapping_noc_country = {
    'CHN': 'China',
    'DEN': 'Denmark',
    'NED': 'Netherlands',
    'USA': 'United States',
    'FIN': 'Finland',
    'NOR': 'Norway',
    'ROU': 'Romania',
    'EST': 'Estonia',
    'FRA': 'France',
    'MAR': 'Morocco',
    'ESP': 'Spain',
    'EGY': 'Egypt',
    'IRI': 'Iran',
    'BUL': 'Bulgaria',
    'ITA': 'Italy',
    'CHA': 'Chad',
    'AZE': 'Azerbaijan',
    'SUD': 'Sudan',
    'RUS': 'Russia',
    'ARG': 'Argentina',
    'CUB': 'Cuba',
    'BLR': 'Belarus',
    'GRE': 'Greece',
    'CMR': 'Cameroon',
    'TUR': 'Turkey',
    'CHI': 'Chile',
    'MEX': 'Mexico',
    'URS': 'Soviet Union',
    'NCA': 'Nicaragua',
    'HUN': 'Hungary',
    'NGR': 'Nigeria',
    'ALG': 'Algeria',
    'KUW': 'Kuwait',
    'BRN': 'Bahrain',
    'PAK': 'Pakistan',
    'IRQ': 'Iraq',
    'UAR': 'United Arab Republic',
    'LIB': 'Lebanon',
    'QAT': 'Qatar',
    'MAS': 'Malaysia',
    'GER': 'Germany',
    'CAN': 'Canada',
    'IRL': 'Ireland',
    'AUS': 'Australia',
    'RSA': 'South Africa',
    'ERI': 'Eritrea',
    'TAN': 'Tanzania',
    'JOR': 'Jordan',
    'TUN': 'Tunisia',
    'LBA': 'Libya',
    'BEL': 'Belgium',
    'DJI': 'Djibouti',
    'PLE': 'Palestine',
    'COM': 'Comoros',
    'KAZ': 'Kazakhstan',
    'BRU': 'Brunei',
    'IND': 'India',
    'KSA': 'Saudi Arabia',
    'SYR': 'Syria',
    'MDV': 'Maldives',
    'ETH': 'Ethiopia',
    'UAE': 'United Arab Emirates',
    'YAR': 'North Yemen',
    'INA': 'Indonesia',
    'PHI': 'Philippines',
    'SGP': 'Singapore',
    'UZB': 'Uzbekistan',
    'KGZ': 'Kyrgyzstan',
    'TJK': 'Tajikistan',
    'EUN': 'Unified Team',
    'JPN': 'Japan',
    'CGO': 'Congo (Brazzaville)',
    'SUI': 'Switzerland',
    'BRA': 'Brazil',
    'FRG': 'West Germany',
    'GDR': 'East Germany',
    'MON': 'Monaco',
    'ISR': 'Israel',
    'URU': 'Uruguay',
    'SWE': 'Sweden',
    'ISV': 'United States Virgin Islands',
    'SRI': 'Sri Lanka',
    'ARM': 'Armenia',
    'CIV': "Côte d'Ivoire",
    'KEN': 'Kenya',
    'BEN': 'Benin',
    'UKR': 'Ukraine',
    'GBR': 'Great Britain',
    'GHA': 'Ghana',
    'SOM': 'Somalia',
    'LAT': 'Latvia',
    'NIG': 'Niger',
    'MLI': 'Mali',
    'AFG': 'Afghanistan',
    'POL': 'Poland',
    'CRC': 'Costa Rica',
    'PAN': 'Panama',
    'GEO': 'Georgia',
    'SLO': 'Slovenia',
    'CRO': 'Croatia',
    'GUY': 'Guyana',
    'NZL': 'New Zealand',
    'POR': 'Portugal',
    'PAR': 'Paraguay',
    'ANG': 'Angola',
    'VEN': 'Venezuela',
    'COL': 'Colombia',
    'BAN': 'Bangladesh',
    'PER': 'Peru',
    'ESA': 'El Salvador',
    'PUR': 'Puerto Rico',
    'UGA': 'Uganda',
    'HON': 'Honduras',
    'ECU': 'Ecuador',
    'TKM': 'Turkmenistan',
    'MRI': 'Mauritius',
    'SEY': 'Seychelles',
    'TCH': 'Czechoslovakia',
    'LUX': 'Luxembourg',
    'MTN': 'Mauritania',
    'CZE': 'Czech Republic',
    'SKN': 'Saint Kitts and Nevis',
    'TTO': 'Trinidad and Tobago',
    'DOM': 'Dominican Republic',
    'VIN': 'Saint Vincent and the Grenadines',
    'JAM': 'Jamaica',
    'LBR': 'Liberia',
    'SUR': 'Suriname',
    'NEP': 'Nepal',
    'MGL': 'Mongolia',
    'AUT': 'Austria',
    'PLW': 'Palau',
    'LTU': 'Lithuania',
    'TOG': 'Togo',
    'NAM': 'Namibia',
    'AHO': 'Netherlands Antilles',
    'ISL': 'Iceland',
    'ASA': 'American Samoa',
    'SAM': 'Samoa',
    'RWA': 'Rwanda',
    'DMA': 'Dominica',
    'HAI': 'Haiti',
    'MLT': 'Malta',
    'CYP': 'Cyprus',
    'GUI': 'Guinea',
    'BIZ': 'Belize',
    'YMD': 'North Yemen',
    'KOR': 'South Korea',
    'THA': 'Thailand',
    'BER': 'Bermuda',
    'ANZ': 'Australasia',
    'SCG': 'Serbia and Montenegro',
    'SLE': 'Sierra Leone',
    'PNG': 'Papua New Guinea',
    'YEM': 'Yemen',
    'IOA': 'Independent Olympic Athletes',
    'OMA': 'Oman',
    'FIJ': 'Fiji',
    'VAN': 'Vanuatu',
    'MDA': 'Moldova',
    'YUG': 'Yugoslavia',
    'BAH': 'Bahamas',
    'GUA': 'Guatemala',
    'SRB': 'Serbia',
    'IVB': 'British Virgin Islands',
    'MOZ': 'Mozambique',
    'CAF': 'Central African Republic',
    'MAD': 'Madagascar',
    'MAL': 'Malaysia',
    'BIH': 'Bosnia and Herzegovina',
    'GUM': 'Guam',
    'CAY': 'Cayman Islands',
    'SVK': 'Slovakia',
    'BAR': 'Barbados',
    'GBS': 'Guinea-Bissau',
    'TLS': 'Timor Leste',
    'COD': 'Congo (Kinshasa)',
    'GAB': 'Gabon',
    'SMR': 'San Marino',
    'LAO': 'Laos',
    'BOT': 'Botswana',
    'ROT': 'Refugee Olympic Athletes',
    'CAM': 'Cambodia',
    'PRK': 'North Korea',
    'SOL': 'Solomon Islands',
    'SEN': 'Senegal',
    'CPV': 'Cape Verde',
    'CRT': 'Croatia',
    'GEQ': 'Equatorial Guinea',
    'BOL': 'Bolivia',
    'SAA': 'Saar',
    'AND': 'Andorra',
    'ANT': 'Antigua and Barbuda',
    'ZIM': 'Zimbabwe',
    'GRN': 'Grenada',
    'HKG': 'Hong Kong',
    'LCA': 'Saint Lucia',
    'FSM': 'Federated States of Micronesia',
    'MYA': 'Myanmar',
    'MAW': 'Malawi',
    'ZAM': 'Zambia',
    'RHO': 'Rhodesia',
    'TPE': 'Chinese Taipei',
    'STP': 'Sao Tome and Principe',
    'MKD': 'Macedonia',
    'BOH': 'Bohemia',
    'TGA': 'Tonga',
    'LIE': 'Liechtenstein',
    'MNE': 'Montenegro',
    'GAM': 'Gambia',
    'COK': 'Cook Islands',
    'ALB': 'Albania',
    'WIF': 'West Indies Federation',
    'SWZ': 'Eswatini',
    'BUR': 'Burundi',
    'NBO': 'North Borneo',
    'BDI': 'Burundi',
    'ARU': 'Aruba',
    'NRU': 'Nauru',
    'VNM': 'Vietnam',
    'VIE': 'Vietnam',
    'BHU': 'Bhutan',
    'MHL': 'Marshall Islands',
    'KIR': 'Kiribati',
    'UNK': 'Unknown',
    'TUV': 'Tuvalu',
    'NFL': 'Newfoundland',
    'KOS': 'Kosovo',
    'SSD': 'South Sudan',
    'LES': 'Lesotho'
    }

In [ ]:
for key, value in mapping_noc_country.items():
    data = data.withColumn(
        "Team",
        when(
            col("NOC") == key,
            value
        ).otherwise(col("Team"))
    )

In [ ]:
# Split
data = data.withColumn("Year_from_Games", split("Games", " ").getItem(0)) \
           .withColumn("Season_from_Games", split("Games", " ").getItem(1))

# Remplacer Year/Season null
data = data.withColumn(
    "Year",
    when(col("Year").isNull(), col("Year_from_Games"))
     .otherwise(col("Year"))
)

data = data.withColumn(
    "Season",
    when(col("Season").isNull(), col("Season_from_Games"))
     .otherwise(col("Season"))
)

# Corriger les Year invalides
data = data.withColumn(
    "Year",
    when(~col("Year").rlike("^[0-9]{4}$"), col("Year_from_Games"))
     .otherwise(col("Year"))
)

# Nettoyer Age
data = data.withColumn(
    "Age",
    when(col("Age").rlike("^[0-9]+$"), col("Age").cast("int"))
     .otherwise(None)
)

# Convertir Year en entier
data = data.withColumn("Year", col("Year").cast("int"))

data = data.withColumn("Birth_Year", col("Year") - col("Age"))

# Drop
data = data.drop("Year_from_Games", "Season_from_Games")

In [ ]:
data = data.withColumn("Height", col("Height").cast("double"))
data = data.withColumn("Weight", col("Weight").cast("double"))

In [ ]:
data = data.withColumn("hasParticipated", lit(1))\
    .withColumn("Medal_flag", when(col("Medal").isNotNull(), 1).otherwise(0))

In [ ]:
data.write.mode("overwrite").format("delta").save("abfss://silver@mohammedstorageacc2.dfs.core.windows.net/athletes_transformed_data/")